# COMP6713 — Notebook 3: Fine-tuning PubMedBERT

This notebook fine-tunes **PubMedBERT** (`microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`)
on our combined medical QA dataset for **extractive question answering**.

PubMedBERT is an encoder-only (BERT-style) transformer pre-trained exclusively on
PubMed biomedical abstracts, making it well-suited for this domain.

> **Note:** Fine-tuning requires a GPU. Run this notebook on the desktop machine (RTX 5070 Ti).
> Inference can be run on CPU after the checkpoint is saved.

In [ ]:
import sys, warnings, torch
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load and Split Data

In [ ]:
from medqa.data.loader import load_all
from medqa.data.preprocessor import split_dataset

records = load_all()
train, test = split_dataset(records)
print(f"Train: {len(train)} | Test: {len(test)}")

## 2. Fine-tune PubMedBERT

In [ ]:
from medqa.models.bert_qa import PubMedBERTQA
from medqa.config import BERT_FINETUNE

print("Training configuration:")
for k, v in BERT_FINETUNE.items():
    print(f"  {k}: {v}")

In [ ]:
model = PubMedBERTQA()

# Check for existing checkpoint
if model.load():
    print("Existing checkpoint found — skipping training.")
    print("Delete checkpoints/bert_qa/ to retrain from scratch.")
else:
    print("Starting fine-tuning...")
    # Use 80% of train for training, 20% for validation
    split_point = int(len(train) * 0.8)
    model.fine_tune(
        train_records = train[:split_point],
        eval_records  = train[split_point:]
    )
    print("Fine-tuning complete!")

## 3. Single Question Inference

In [ ]:
question = "What is the first-line treatment for Type 2 diabetes?"
context  = test[0]['context']

result = model.predict(question, context)
print(f"Question : {question}")
print(f"Answer   : {result['predicted_answer']}")
print(f"Score    : {result['score']:.4f}")

## 4. Batch Evaluation

In [ ]:
from medqa.evaluation.metrics import evaluate, print_results

test_subset = test[:200]
questions   = [r['question'] for r in test_subset]
contexts    = [r['context']  for r in test_subset]
golds       = [r['answer']   for r in test_subset]

preds_data = model.batch_predict(questions, contexts)
preds      = [p['predicted_answer'] for p in preds_data]

results = evaluate(preds, golds, use_bertscore=False)
print_results(results, model_name='Fine-tuned PubMedBERT')

## 5. Error Analysis

In [ ]:
from medqa.evaluation.qualitative import analyse_errors
from medqa.evaluation.metrics import token_f1

eval_records = []
for r, pred in zip(test_subset, preds):
    eval_records.append({
        'question':         r['question'],
        'predicted_answer': pred,
        'gold_answer':      r['answer'],
        'context':          r['context'],
        'token_f1':         token_f1(pred, r['answer']),
    })

summary = analyse_errors(
    eval_records,
    f1_threshold=0.3,
    output_path='../data/processed/bert_errors.json'
)

## 6. Comparison with Baseline

In [ ]:
from medqa.models.baseline import TFIDFBaseline
import nltk; nltk.download('punkt_tab', quiet=True)

bl = TFIDFBaseline()
bl.load()
bl_preds = [bl.predict(q)['predicted_answer'] for q in questions]
bl_results = evaluate(bl_preds, golds, use_bertscore=False)

print("Model Comparison (n=200)")
print(f"{'Model':<25} {'EM':>8} {'F1':>8}")
print("-" * 45)
print(f"{'TF-IDF Baseline':<25} {bl_results['mean_em']:>8.4f} {bl_results['mean_f1']:>8.4f}")
print(f"{'Fine-tuned BERT':<25} {results['mean_em']:>8.4f} {results['mean_f1']:>8.4f}")
improvement_f1 = (results['mean_f1'] - bl_results['mean_f1']) / max(bl_results['mean_f1'], 1e-9) * 100
print(f"\nBERT improves F1 by {improvement_f1:.1f}% over baseline")

## Summary

PubMedBERT is fine-tuned as an **extractive QA model** — it predicts the start and
end token positions of the answer span within the provided context.

**Key design choices:**
- Base model: `BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext` — pre-trained on biomedical text
- Truncation strategy: `longest_first` — handles long contexts gracefully
- Manual forward pass inference — more control than HuggingFace pipeline

**Expected results:**
- Exact Match ≈ 0.51 (significant improvement over baseline)
- Token F1 ≈ 0.55
- BERTScore ≈ 0.89